In [4]:
# Cài đặt thư viện ultralytics nếu máy chưa có
!pip install ultralytics opencv-python pandas matplotlib scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import os

# Đường dẫn tới file CSV dựa theo cây thư mục của bạn
csv_path = "archive/data/train_solution_bounding_boxes (1).csv"
if not os.path.exists(csv_path):
    filenames = {
        "train_solution_bounding_boxes (1).csv",
        "train_solution_bounding_boxes.csv",
    }
    matches = [
        os.path.join(root, filename)
        for root, _, files in os.walk(".")
        for filename in files
        if filename in filenames
    ]

    if not matches:
        raise FileNotFoundError(
            f"CSV not found. Current working directory: {os.getcwd()}. "
            "Check that the dataset is extracted and the filename is correct."
        )

    csv_path = matches[0]
    print(f"Using CSV: {csv_path}")

df = pd.read_csv(csv_path)

print(f"Tổng số dòng trong file CSV: {len(df)}")
print("5 dòng đầu tiên của dữ liệu:")
display(df.head())

Using CSV: .\train_solution_bounding_boxes (1).csv
Tổng số dòng trong file CSV: 559
5 dòng đầu tiên của dữ liệu:


,image,xmin,ymin,xmax,ymax
0,vid_4_1000.jpg,281.259045,187.035071,327.727931,223.225547
1,vid_4_10000.jpg,15.163531,187.035071,120.329957,236.430180
2,vid_4_10040.jpg,239.192475,176.764801,361.968162,236.430180
3,vid_4_10020.jpg,496.483358,172.363256,630.020260,231.539575
4,vid_4_10060.jpg,16.630970,186.546010,132.558611,238.386422


In [8]:
import yaml

# Nội dung file cấu hình dataset cho YOLO
data_config = {
    'path': os.path.abspath("archive/data/yolo_dataset"),  # Đường dẫn tuyệt đối tới thư mục dataset
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,                  # Số lượng lớp (chỉ có 1 lớp là 'car')
    'names': ['car']          # Tên của lớp
}

# Lưu thành file data.yaml
yaml_path = "archive/data/data.yaml"
data_dir = os.path.abspath(".")
data_config["path"] = os.path.join(data_dir, "yolo_dataset")
yaml_path = os.path.join(data_dir, "data.yaml")

with open(yaml_path, "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"Đã tạo file cấu hình thành công tại: {yaml_path}")

Đã tạo file cấu hình thành công tại: c:\Users\ACER\Desktop\deep learning\lab dl\archive\data\data.yaml


In [12]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import random
import shutil
model=YOLO("yolov8n.pt")  # Sử dụng mô hình YOLOv8 nano đã được huấn luyện trước
train_images = os.path.join(data_config["path"], "images", "train")
val_images = os.path.join(data_config["path"], "images", "val")

if not os.path.isdir(train_images):
    dataset_root = Path(data_config["path"])
    wanted = {str(name).casefold() for name in df["image"].dropna().unique()}
    image_sources = {}

    # Search for the CSV's images under the dataset directory.
    for search_root in {Path(data_dir).resolve(), Path.cwd().resolve()}:
        for root, dirs, files in os.walk(search_root):
            # Avoid searching the output dataset being created.
            dirs[:] = [
                d for d in dirs
                if Path(root, d).resolve() != dataset_root.resolve()
            ]
            for filename in files:
                if filename.casefold() in wanted:
                    image_sources.setdefault(filename.casefold(), Path(root) / filename)
            if len(image_sources) == len(wanted):
                break

    missing = sorted(wanted - image_sources.keys())
    if missing:
        raise FileNotFoundError(
            f"Could not find {len(missing)} source images under {data_dir}. "
            f"Examples: {missing[:10]}. Extract the images into that directory, "
            "then rerun this cell."
        )

    image_names = sorted(df["image"].dropna().astype(str).unique())
    random.Random(42).shuffle(image_names)
    val_count = max(1, round(len(image_names) * 0.2)) if len(image_names) > 1 else 0
    val_names = set(image_names[:val_count])
    train_names = set(image_names[val_count:])

    for split, names in (("train", train_names), ("val", val_names)):
        images_dir = dataset_root / "images" / split
        labels_dir = dataset_root / "labels" / split
        images_dir.mkdir(parents=True, exist_ok=True)
        labels_dir.mkdir(parents=True, exist_ok=True)

        for image_name in names:
            source = image_sources[image_name.casefold()]
            image = cv2.imread(str(source))
            if image is None:
                raise ValueError(f"Unable to read image: {source}")

            height, width = image.shape[:2]
            shutil.copy2(source, images_dir / source.name)

            rows = df[df["image"].astype(str) == image_name]
            label_lines = []
            for _, row in rows.iterrows():
                x1 = max(0, min(float(row["xmin"]), width))
                y1 = max(0, min(float(row["ymin"]), height))
                x2 = max(0, min(float(row["xmax"]), width))
                y2 = max(0, min(float(row["ymax"]), height))
                if x2 <= x1 or y2 <= y1:
                    continue

                xc = ((x1 + x2) / 2) / width
                yc = ((y1 + y2) / 2) / height
                box_w = (x2 - x1) / width
                box_h = (y2 - y1) / height
                label_lines.append(f"0 {xc:.6f} {yc:.6f} {box_w:.6f} {box_h:.6f}")

            (labels_dir / f"{source.stem}.txt").write_text(
                "\n".join(label_lines), encoding="utf-8"
            )

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_config, f, sort_keys=False)

    print(f"Created YOLO dataset at: {dataset_root}")

if not os.path.isdir(val_images):
    print("Warning: images/val is missing; using images/train for validation.")
    data_config["val"] = "images/train"
    with open(yaml_path, "w") as f:
        yaml.safe_dump(data_config, f, sort_keys=False)

results = model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    batch=8,
    device="cpu",
    name="car_detector_run"
)
print ("Huan luyen hoan tat ! Trong so tot nhat duoc luu tai thu muc runs/detect/car_detector_run")

Created YOLO dataset at: c:\Users\ACER\Desktop\deep learning\lab dl\archive\data\yolo_dataset
New https://pypi.org/project/ultralytics/8.4.215 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.184  Python-3.12.7 torch-2.14.1+cpu CPU (12th Gen Intel Core i5-12450H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\ACER\Desktop\deep learning\lab dl\archive\data\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64

In [ ]:
import cv2
from PIL import Image
import glob

# Load trọng số sau khi train xong
weights_path = Path(results.save_dir) / "weights" / "best.pt"

if not weights_path.is_file():
    raise FileNotFoundError(
        f"Model weights not found at {weights_path}. "
        "Run the training cell to completion first."
    )

trained_model = YOLO(str(weights_path))

# Lấy thử một vài ảnh từ thư mục testing_images của bạn để test
test_images = glob.glob("archive/data/testing_images/*.jpg")[:3]

for img_path in test_images:
    # Chạy dự đoán
    results = trained_model(img_path)
    
    # Hiển thị ảnh kết quả trực tiếp trong Notebook
    for r in results:
        im_bgr = r.plot()  # Vẽ bounding box lên ảnh
        im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(8, 6))
        plt.imshow(im_rgb)
        plt.axis('off')
        plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'runs\\detect\\car_detector_run\\weights\\best.pt'